<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/PESCA/Pesca_SMN_UCA_Long.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pesca San Martín y Ucayali · LONG

Lleva los Excel de DIREPRO San Martín (uno por mes, en kg) y DIREPRO Ucayali (acumulado del año, en t) a
`SMN_PESCA_LONG` y `UCA_PESCA_LONG`, con el mismo criterio que los LONG que ya existen: las especies y su
clasificación (`ESPECIE_OK`, `ESPECIES_CORREGIDO`, `ESPECIES_OK`) se toman del propio histórico, y el VBP es
volumen × precio base 2007. Cada corrida reemplaza los periodos que traen los crudos y conserva el resto.
Detalle en `PESCA/pesca.md`.

In [ ]:
# ═══ BLOQUE 1 · PARÁMETROS (revisar cada mes) ═══
# Carpetas en Google Drive (se monta en el Bloque 3). Un LONG que no existe se crea; si existe, se respalda y se actualiza.
CARPETA_SMN   = "/content/drive/MyDrive/PESCA/SAN_MARTIN/CRUDOS"           # SMN_PESCA_<AÑO>-<MES>.xlsx, uno por mes
RUTA_SMN_LONG = "/content/drive/MyDrive/PESCA/SAN_MARTIN/SMN_PESCA_LONG.xlsx"
CARPETA_UCA   = "/content/drive/MyDrive/PESCA/UCAYALI/CRUDOS"              # ESTADISTICA_PESQUERA_DIREPRO_UCA_*.xlsx (acumulado del año)
RUTA_UCA_LONG = "/content/drive/MyDrive/PESCA/UCAYALI/UCA_PESCA_LONG.xlsx"

# Precio base 2007, en miles de soles por tonelada
PRECIOS = {'FRESCO': 3.27504, 'SALPRESO': 3.01589, 'SECO SALADO': 2.98778}

# Especies nuevas (que no están en el histórico): especie principal por origen en San Martín; el resto va a "Otras"
SMN_PRINCIPALES = {'Acuicultura': {'CAMARÓN': 'Camarón gigante', 'GAMITANA': 'Gamitana', 'PACO': 'Paco',
                                   'PAICHE': 'Paiche', 'TILAPIA': 'Tilapia'},
                   'Extracción de ríos': {'BOQUICHICO': 'Boquichico'}}
SMN_ORIGEN = {'ACUICULTURA CONTINENTAL': 'Acuicultura', 'CONTINENTAL': 'Extracción de ríos',
              'MARÍTIMO': 'Extracción de ríos'}
SMN_ESTADOS = {'FRESCO': 'FRESCO', 'FRESCO REFRIGERADO': 'FRESCO', 'SALPRESO': 'SALPRESO', 'SECO SALADO': 'SECO SALADO'}
SMN_COLUMNAS = ['AÑO', 'MES', 'DEPARTAMENTO', 'ORIGEN', 'ESTADO', 'ESTADO_OK', 'ESPECIE', 'ESPECIE_OK', 'VOLUMEN', 'VBP']

# Ucayali: hoja del crudo por tipo, y precio de cada tipo (el agregado y los ornamentales no llevan VBP)
UCA_HOJAS = {'FRESCO': 'totalfresco', 'SECO-SALADO': 'totalsecosalado', 'ORNAMENTAL': 'ornamentales',
             'AGREGADO': 'totaldepescado'}
UCA_PRECIOS = {'FRESCO': PRECIOS['FRESCO'], 'SECO-SALADO': PRECIOS['SECO SALADO'], 'ORNAMENTAL': 0, 'AGREGADO': 0}
UCA_COLUMNAS = ['AÑO', 'MES', 'TIPO', 'ESPECIES', 'ESPECIES_CORREGIDO', 'ESPECIES_OK', 'VOLUMEN', 'VBP']


In [ ]:
# ═══ BLOQUE 2 · FUNCIONES (no hace falta tocar) ═══
import re, shutil, unicodedata
from datetime import datetime
from pathlib import Path
import pandas as pd

def _clave(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())


def _texto(v):
    return ' '.join(str(v).split())


def respaldar(ruta):
    """Copia el LONG antes de sobrescribirlo (sus tablas dinámicas no se conservan en el archivo nuevo)."""
    ruta = Path(ruta)
    if ruta.exists():
        copia = ruta.with_name(f'{ruta.stem}_respaldo_{datetime.now():%Y%m%d_%H%M%S}{ruta.suffix}')
        shutil.copy2(ruta, copia)
        print(f'💾 Respaldo: {copia.name}')


# ─── San Martín ───
def periodo_smn(ruta):
    """Año y mes del nombre del archivo (SMN_PESCA_2026-08.xlsx): el crudo de San Martín no los trae adentro."""
    m = re.search(r'(20\d{2})[-_ ]?(0[1-9]|1[0-2])(?!\d)', Path(ruta).stem)
    if m:
        return int(m.group(1)), int(m.group(2))
    raise ValueError(f'{Path(ruta).name}: el nombre debe traer el año y el mes (por ejemplo SMN_PESCA_2026-08.xlsx)')


def long_smn(ruta, historico=None):
    """LONG de un mes desde el crudo de DIREPRO San Martín (volúmenes en kg)."""
    xls = pd.ExcelFile(ruta)
    anio, mes = periodo_smn(ruta)
    d = pd.read_excel(xls, xls.sheet_names[0])
    d.columns = [_texto(c).upper() for c in d.columns]
    mapa = {}
    if historico is not None:
        mapa = {(o, _texto(e)): ok for o, e, ok in historico[['ORIGEN', 'ESPECIE', 'ESPECIE_OK']].drop_duplicates().values}
    filas = []
    for _, r in d.iterrows():
        proc, depto, esp = _texto(r['PROCEDENCIA']).upper(), r['DEPARTAMENTO'], _texto(r['ESPECIE'])
        if proc == 'TOTAL' or pd.isna(depto) or proc not in SMN_ORIGEN:
            continue
        depto = _texto(depto).upper()
        encabezado = int(pd.to_numeric(r['ENCABEZADO'], errors='coerce') or 0) == 1
        if depto == 'SAN MARTÍN' and not encabezado:          # especies de San Martín: solo celdas con volumen
            origen, especie = SMN_ORIGEN[proc], esp
            estados = [(e, r.get(e)) for e in SMN_ESTADOS if pd.to_numeric(r.get(e), errors='coerce') > 0]
            dep = 'SAN MARTÍN'
        elif depto != 'SAN MARTÍN' and encabezado:             # lo que llega de otros departamentos: su total
            origen, especie, dep = 'Extracción de ríos', proc, 'Otros'
            estados = [(e, r.get(e)) for e in SMN_ESTADOS]
        else:
            continue
        for estado, kg in estados:
            filas.append((dep, origen, estado, especie, (pd.to_numeric(kg, errors='coerce') or 0) / 1000))
    df = pd.DataFrame(filas, columns=['DEPARTAMENTO', 'ORIGEN', 'ESTADO', 'ESPECIE', 'VOLUMEN']).fillna({'VOLUMEN': 0})
    df['ESTADO_OK'] = df.ESTADO.map(SMN_ESTADOS)
    df['ESPECIE_OK'] = [mapa.get((o, e)) or SMN_PRINCIPALES.get(o, {}).get(e.upper(), 'Otras')
                        for o, e in zip(df.ORIGEN, df.ESPECIE)]
    df['VBP'] = df.VOLUMEN * df.ESTADO_OK.map(PRECIOS) * df.DEPARTAMENTO.eq('SAN MARTÍN')
    df['AÑO'], df['MES'] = anio, mes
    nuevas = sorted({e for o, e in zip(df.ORIGEN, df.ESPECIE) if mapa and (o, e) not in mapa})
    if nuevas:
        print(f'  ℹ Especies que no estaban en el histórico: {", ".join(nuevas)}')
    return df[SMN_COLUMNAS]


# ─── Ucayali ───
def _cuadro_uca(xls, hoja):
    """Especie × mes de un cuadro de DIREPRO Ucayali (fila ESPECIES con las fechas AAAA-MM-01)."""
    d = pd.read_excel(xls, hoja, header=None)
    f, c = next((i, j) for i in range(15) for j in range(3) if _texto(d.iat[i, j]).upper() == 'ESPECIES')
    meses = {}
    for j, v in d.iloc[f].items():
        m = re.match(r'(20\d{2})-(\d{2})', str(v))
        if m:
            meses[j] = (int(m.group(1)), int(m.group(2)))
    filas = []
    for i in range(f + 1, len(d)):
        esp = d.iat[i, c]
        if pd.isna(esp) or _texto(esp).upper().startswith('TOTAL'):
            break
        for j, (a, m) in meses.items():
            filas.append((a, m, _texto(esp), pd.to_numeric(d.iat[i, j], errors='coerce')))
    return pd.DataFrame(filas, columns=['AÑO', 'MES', 'ESPECIES', 'VOLUMEN']).fillna({'VOLUMEN': 0})


def long_uca(ruta, historico=None):
    """LONG del año desde el crudo acumulado de DIREPRO Ucayali (toneladas; ornamentales en unidades)."""
    xls = pd.ExcelFile(ruta)
    mapa = {}
    if historico is not None:
        h = historico.assign(E=historico.ESPECIES.map(_texto))
        mapa = {(t, e): (c, ok) for t, e, c, ok in
                h[['TIPO', 'E', 'ESPECIES_CORREGIDO', 'ESPECIES_OK']].drop_duplicates(['TIPO', 'E']).values}
    partes, nuevas = [], set()
    for tipo, clave in UCA_HOJAS.items():
        hoja = next((h for h in xls.sheet_names if _clave(h) == clave), None)
        if hoja is None:
            raise ValueError(f'No se encontró la hoja de {tipo} en {Path(ruta).name}')
        t = _cuadro_uca(xls, hoja)
        if tipo == 'ORNAMENTAL':
            t = t[t.VOLUMEN != 0]
        t['TIPO'] = tipo
        defecto = lambda e: (e.title(), e.title() if tipo in ('SECO-SALADO', 'ORNAMENTAL') else 'Otros')
        pares = [mapa.get((tipo, e)) or defecto(e) for e in t.ESPECIES]
        nuevas |= {(tipo, e) for e in t.ESPECIES if mapa and (tipo, e) not in mapa}
        t['ESPECIES_CORREGIDO'], t['ESPECIES_OK'] = [p[0] for p in pares], [p[1] for p in pares]
        t['VBP'] = t.VOLUMEN * UCA_PRECIOS[tipo]
        partes.append(t)
    if nuevas:
        print('  ℹ Especies que no estaban en el histórico:', ', '.join(f'{e} ({t})' for t, e in sorted(nuevas)))
    return pd.concat(partes, ignore_index=True)[UCA_COLUMNAS]


def unir(nuevo, historico, claves):
    """Reemplaza en el histórico los periodos (claves) que trae lo nuevo, en el mismo lugar del archivo; un periodo
    que no estaba va al final. El resto se conserva en su orden."""
    if historico is None or not len(historico):
        return nuevo.reset_index(drop=True)
    quitar = set(map(tuple, nuevo[claves].drop_duplicates().values))
    sale = pd.Series([tuple(k) in quitar for k in historico[claves].values], index=historico.index)
    pos = sale.values.argmax() if sale.any() else len(historico)
    antes, despues = historico.iloc[:pos], historico.iloc[pos:]
    return pd.concat([antes[~sale.iloc[:pos].values], nuevo, despues[~sale.iloc[pos:].values]], ignore_index=True)


def resumen_vbp(df, por):
    """VBP por año y mes (filas) y estado o tipo (columnas)."""
    return df.pivot_table(index=['AÑO', 'MES'], columns=por, values='VBP', aggfunc='sum', fill_value=0,
                          margins=True, margins_name='TOTAL')


In [ ]:
# ═══ BLOQUE 3 · SAN MARTÍN ═══
from google.colab import drive
drive.mount('/content/drive')


def crudos(carpeta):
    return sorted(p for p in Path(carpeta).glob('*.xlsx') if not p.name.startswith('~$'))


def actualizar(carpeta, ruta_long, hoja, generar, claves, por):
    lista = crudos(carpeta)
    if not lista:
        print(f'⏭ No hay .xlsx en {carpeta}')
        return None
    long_ = pd.read_excel(ruta_long, sheet_name=hoja) if Path(ruta_long).exists() else None
    for ruta in lista:
        nuevo = generar(ruta, long_)
        periodos = sorted('-'.join(f'{int(v):02d}' for v in k) for k in nuevo[claves].drop_duplicates().values)
        print(f'📄 {ruta.name} → {", ".join(periodos)} · {len(nuevo)} filas · VBP {nuevo.VBP.sum():,.3f} mil S/ de 2007')
        long_ = unir(nuevo, long_, claves)
    respaldar(ruta_long)
    Path(ruta_long).parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(ruta_long, engine='openpyxl') as xw:
        long_.to_excel(xw, sheet_name=hoja, index=False)
        resumen_vbp(long_, por).to_excel(xw, sheet_name='VBP')
    print(f'✅ {ruta_long}: {len(long_):,} filas, {long_["AÑO"].min()}–{long_["AÑO"].max()}')
    return long_


smn = actualizar(CARPETA_SMN, RUTA_SMN_LONG, 'SMN_PESCA', long_smn, ['AÑO', 'MES'], 'ESTADO_OK')
if smn is not None:
    ult = smn[(smn['AÑO'] == smn['AÑO'].max())]
    display(ult[ult.MES == ult.MES.max()].groupby(['DEPARTAMENTO', 'ESTADO_OK'])[['VOLUMEN', 'VBP']].sum().round(3))


In [ ]:
# ═══ BLOQUE 4 · UCAYALI ═══
uca = actualizar(CARPETA_UCA, RUTA_UCA_LONG, 'UCA_PESCA', long_uca, ['AÑO'], 'TIPO')
if uca is not None:
    ult = uca[uca['AÑO'] == uca['AÑO'].max()]
    display(ult.pivot_table(index='MES', columns='TIPO', values='VOLUMEN', aggfunc='sum').round(3))
